In [ ]:
import torch
import yaml
import matplotlib.pyplot as plt
import numpy as np
import wandb
from copy import deepcopy
from dotenv import load_dotenv
from helper.utils import train_loop, test_loop
from helper import load_default_dataloaders, load_ObjDet_model

In [ ]:
torch.manual_seed(42)

In [ ]:
load_dotenv() 

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

## Loading Object Detection Model and Dataset

In [ ]:
dataset_path = "../kaggle/Traffic_Sign/car"
batch_size = 64
num_workers = 0 # multiple workers will not work in a ipynb on macOS

In [ ]:
# load class names from yaml file

with open(f"{dataset_path}/data.yaml", "r") as f:
    class_names = yaml.safe_load(f)["names"]

In [ ]:
# building the model with the new object detection head, getting the test dataloader from the weights preprocessing and the new input image size after the preprocessing transformations

model, test_transform, image_size = load_ObjDet_model(num_classes=len(class_names), frozen=True, device=device)

In [ ]:
train_dataloader, val_dataloader, _ = load_default_dataloaders(
    batch_size=batch_size,
    num_workers=num_workers, 
    image_size=image_size, 
    test_transform=test_transform, 
    dataset_path=dataset_path
)

## Defining Loss Function Parameters

In [ ]:
# defining weights for the loss calculation
no_obj_w = 0.2 
loc_w = 4
obj_w = 12
class_w = 1
loc_l2_factor = 1

## Training model head on whole Dataset

In [ ]:
def train_model(model, train_dataloader, val_dataloader, optimizer, wandb_project_name, wandb_config, epochs):
    train_losses = []
    val_losses = []
    best_model_dict = None
    best_val_loss = float("inf")

    labels = ["Total_Loss", "Objectness_Loss", "Classification_Loss", "Localization_Loss"]

    with wandb.init(project=wandb_project_name, config=wandb_config) as run:
        run.watch(model)

        for epoch in range(epochs):
            print(f"-------------------------------\nEpoch {epoch+1}")

            train_batch_losses = train_loop(model, train_dataloader, optimizer)
            val_batch_losses = test_loop(model, val_dataloader)


            mean_train_losses = [np.array(l).mean() for l in zip(*train_batch_losses)]
            train_losses.append(mean_train_losses)
            total_mean_train_batch_loss = mean_train_losses[0]

            mean_val_losses = [np.array(l).mean() for l in zip(*val_batch_losses)]
            val_losses.append(mean_val_losses)
            total_mean_val_batch_loss = mean_val_losses[0]

            if total_mean_val_batch_loss < best_val_loss:
                best_val_loss = total_mean_val_batch_loss
                best_model_dict = deepcopy(model.state_dict())
                run.summary["best_val_loss"] = best_val_loss    # save the validition loss so it can be seen in the dashboard

            print("Average Train Loss:", total_mean_train_batch_loss)
            print("Average Validation Loss:", total_mean_val_batch_loss)

            # because wandb.log() can only log scalars, we convert the list into a dict which will be then logged
            log_dict = {"epoch": epoch}
            for label, train_l, val_l in zip(labels, mean_train_losses, mean_val_losses):
                # create key-value pairs for each metric for the dict
                log_dict[f"train/{label}"] = train_l
                log_dict[f"val/{label}"] = val_l
            run.log(log_dict)


    torch.save(best_model_dict, f"best_models/TrafficSign_ObjDet_head_{best_val_loss:.4f}.pth")

    return train_losses, val_losses, best_val_loss


In [ ]:
learning_rate = 1e-3  # despite overshooting gradients quickly, especially on small objects, with warmup and lr decay later in training, a higher lr works well
epochs = 50
warmup_epochs = epochs * 0.2
weight_decay = 1e-4
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

warmup = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda epoch: min(1, (epoch + 1) / warmup_epochs))
lr_decay = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs-warmup_epochs)
lr_scheduler = torch.optim.lr_scheduler.SequentialLR(optimizer, [warmup, lr_decay], [warmup_epochs])

# localization loss neads a smaller learning rate at the beginning, otherwise training will get unstable due to large gradients at the beginning and then small gradients due to small boxes with low IoU

In [ ]:
# train new head of the model (with frozen backbone) on the custom dataset

wandb_config = {
    "epochs": epochs,
    "learning_rate": learning_rate,
    "batch_size": batch_size,
    "no_object_weight": no_obj_w,
    "loc_weight": loc_w,
    "objectness_weight": obj_w,
    "wh_l2_factor": loc_l2_factor,
    "optimizer": type(optimizer).__name__,
    "lr_scheduler": lr_scheduler.state_dict()
}

train_losses, val_losses, best_val_loss = train_model(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    wandb_project_name="object-detection-resnet50",
    wandb_config=wandb_config, 
    epochs=epochs
)

print("Model reached best validation loss of:", best_val_loss)

In [ ]:
def plot_loss_curves(train_losses, val_losses):
    fig, axs = plt.subplots(1, 2, figsize=(15, 5))

    labels = ["Objectness Loss", "Classification Loss", "Localization Loss"]
    colors = ["tab:blue", "tab:orange", "tab:green"]

    # Left: total loss
    axs[0].plot([epoch_losses[0] for epoch_losses in train_losses], label="Train Total", color="tab:blue")
    axs[0].plot([epoch_losses[0] for epoch_losses in val_losses], label="Val Total", color="tab:blue", linestyle="--")
    axs[0].set_title("Total Loss")
    axs[0].set_xlabel("Epochs")
    axs[0].set_ylabel("Loss")
    axs[0].legend()
    axs[0].grid(True, alpha=0.3)

    # Right: component losses
    for idx, (lbl, color) in enumerate(zip(labels, colors), start=1):
        axs[1].plot([epoch_losses[idx] for epoch_losses in train_losses], label=f"Train {lbl}", color=color)
        axs[1].plot([epoch_losses[idx] for epoch_losses in val_losses], label=f"Val {lbl}", color=color, linestyle="--")

    axs[1].set_title("Component Losses")
    axs[1].set_xlabel("Epochs")
    axs[1].set_ylabel("Loss")
    axs[1].legend()
    axs[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_loss_curves(train_losses, val_losses)